# 10 — Passing-Network Vulnerability: Part 3, Targeted vs. Eligible-Pool Disruption

*2018 & 2022 FIFA World Cup · StatsBomb event data*

**Central question:** for each team-match, does removing the most structurally damaging
player cause substantially more damage than removing a typical (eligible) player from the
same observed network? This distinguishes **distributed/robust** networks (many removals
cause similar damage) from **concentrated/vulnerable** ones (one player's removal is
unusually damaging).

**Scope:** Part 3 only. No two-player removal, no tournament rankings, no outcome
analysis, no visualizations. Builds on the validated Part 1 baseline and Part 2
single-player removal simulation.

**Primary metric:** `efficiency_damage`. `density_damage` is excluded (frequently rises
after removal purely from the shrinking node-count denominator) and
`largest_component_damage` is excluded (near-zero in 98.9% of Part 2 removals).
`progressive_capacity_damage` is kept as a secondary, football-specific metric. No
composite score yet.

**Methodology note:** the "typical player" baseline is the *exact* mean/std over the
fully-observed eligible pool for each team-match — Part 2 already computed damage for
every eligible player, so this is a population statistic, not a Monte Carlo estimate.
Sampling would only add noise to a quantity already known exactly.

**A note on what NOT to headline:** the targeted player is *defined* as the eligible-pool
maximum, so "targeted damage exceeds the pool mean" is mechanically guaranteed — it is not
a finding. The real question is whether that maximum belongs to an ordinary hub (highest
pass volume, highest betweenness) or not. That's what the vulnerability-concentration
check (Section 3 below) actually tests.

**Interpretation:** this measures how unusually dependent the *observed* network is on
its most structurally important participant, relative to other meaningful participants —
not a behavioral counterfactual about what would happen if an opponent man-marked them.

In [1]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import network_vulnerability as nv

PROC_DIR = Path('../data/processed')

player_network_baseline = pd.read_csv(PROC_DIR / 'player_network_baseline.csv')
single_player_removal = pd.read_csv(PROC_DIR / 'single_player_removal.csv')

## 1. Eligibility rule for the baseline pool

Before applying any rule, check how many players each candidate threshold would exclude.

In [2]:
nv.report_eligibility_thresholds(single_player_removal)

,rule,eligible,excluded,pct_excluded
0,involvement >= 3,3686,72,0.019159
1,involvement >= 5,3632,126,0.033528
2,involvement >= 10,3415,343,0.091272
3,share >= 1%,3490,268,0.071315
4,share >= 2%,3168,590,0.156998
5,CHOSEN: involvement >= 5 OR share >= 2%,3632,126,0.033528


**Chosen rule:** `total_pass_involvement >= 5` OR `share of team-match involvement >= 2%`.
This excludes 126 / 3,758 players (3.4%) — mostly late substitutes with only a touch or
two — while keeping every meaningful contributor. The same eligible pool is used for both
identifying the targeted player and the baseline statistics, as required.

In [3]:
robustness_path = PROC_DIR / 'team_match_robustness.csv'
if robustness_path.exists():
    team_match_robustness = pd.read_csv(robustness_path)
else:
    team_match_robustness = nv.build_team_match_robustness(single_player_removal, player_network_baseline)
    team_match_robustness.to_csv(robustness_path, index=False)

print(team_match_robustness.shape)
team_match_robustness.head(10)

(256, 19)


,match_id,year,team,total_players,eligible_players,targeted_player,targeted_efficiency_damage,targeted_progressive_capacity_damage,targeted_pass_involvement,targeted_betweenness,targeted_pass_involvement_rank,eligible_efficiency_damage_mean,eligible_efficiency_damage_std,eligible_progressive_damage_mean,targeted_excess_efficiency_damage,targeted_standardized_gap,targeted_to_random_ratio,targeted_damage_percentile,top1_top2_gap
0,7525,2018,Russia,14,14,Yuri Zhirkov,0.019427,0.224719,53,0.264286,2,2.156013e-17,0.011924,0.142857,0.019427,1.629237,NaN,100.0,0.007157
1,7525,2018,Saudi Arabia,14,12,Salman Mohammed Al Faraj,0.048799,0.281553,140,0.176816,1,8.946446e-03,0.019775,0.165858,0.039852,2.015268,5.454545,100.0,0.026276
2,7529,2018,Croatia,14,14,Ivan Perišić,0.017932,0.225225,68,0.063034,4,2.230359e-17,0.018025,0.142857,0.017932,0.994850,NaN,100.0,0.000000
3,7529,2018,Nigeria,13,13,Oghenekaro Etebo,0.025216,0.235849,118,0.105429,1,-3.843080e-17,0.018051,0.153846,0.025216,1.396908,NaN,100.0,0.008626
4,7530,2018,Australia,14,13,Josh Risdon,0.040016,0.263636,57,0.447650,6,3.197657e-03,0.020340,0.153147,0.036818,1.810110,12.514019,100.0,0.016317
5,7530,2018,France,14,14,Corentin Tolisso,0.022727,0.176471,113,0.093162,1,-2.230359e-17,0.019630,0.142857,0.022727,1.157767,NaN,100.0,0.000000
6,7531,2018,Argentina,14,14,Faustino Marcos Alberto Rojo,0.021739,0.204678,134,0.073642,3,-7.186712e-17,0.016203,0.142857,0.021739,1.341641,NaN,100.0,0.000000
7,7531,2018,Iceland,14,13,Emil Hallfreðsson,0.034661,0.085106,37,0.155177,2,4.680054e-03,0.018126,0.152209,0.029981,1.654056,7.406061,100.0,0.015487
8,7532,2018,Denmark,14,13,Thomas Delaney,0.055921,0.193878,61,0.113624,4,5.735493e-03,0.018949,0.153846,0.050186,2.648502,9.750000,100.0,0.038377
9,7532,2018,Peru,14,14,Christian Alberto Cueva Bravo,0.026490,0.342342,99,0.085684,2,1.839588e-04,0.018754,0.142857,0.026306,1.402703,144.000000,100.0,0.000000


## 2. Validation checks

In [4]:
nv.run_robustness_validation_checks(team_match_robustness, single_player_removal, player_network_baseline)

TARGETED vs. ELIGIBLE-POOL DISRUPTION — VALIDATION CHECKS

1. Team-matches with zero eligible players: 0 (256 / 256 team-matches produced a robustness row)

2. Targeted player always in the eligible set: 256 / 256
3. Targeted efficiency damage equals max eligible efficiency damage: 256 / 256

4. Spot-checked 25 team-matches: eligible-pool mean/std exactly match an independent recomputation in 25 / 25



5. Re-running produces identical results (fully deterministic, no sampling): True

6. Infinities — ratio: 0, excess: 0. Ratio undefined (NaN, near-zero eligible-mean denominator): 105

7. Team-matches with a positive eligible-mean baseline: 151 (105 excluded — negative/near-zero baseline makes '% above baseline' ill-defined)
   targeted damage >25% above eligible mean: 151 / 151 (100.0%)
   targeted damage >50% above eligible mean: 151 / 151 (100.0%)
   targeted damage >100% above eligible mean: 151 / 151 (100.0%)
   (Expected by construction, since targeted = max. Descriptive only,
   not a significance test, and not the headline finding — see Section 3.)


## 3. Is vulnerability actually concentrated?

**The headline finding is Q3-Q5 below, not the targeted-vs-mean comparison** (which is
mechanical by construction — see the methodology note above).

In [5]:
n = len(team_match_robustness)

# Q3: how often is the targeted player also #1 by raw pass involvement?
top1_by_involvement = (team_match_robustness['targeted_pass_involvement_rank'] == 1).sum()
print(f"Q3. Targeted player is also #1 by pass involvement: "
      f"{top1_by_involvement} / {n} ({top1_by_involvement/n:.1%})")

# Q4: how often is the targeted player NOT in the top 3 by pass involvement?
not_top3 = (team_match_robustness['targeted_pass_involvement_rank'] > 3).sum()
print(f"Q4. Targeted player is NOT in the top 3 by pass involvement: "
      f"{not_top3} / {n} ({not_top3/n:.1%})")

Q3. Targeted player is also #1 by pass involvement: 69 / 256 (27.0%)
Q4. Targeted player is NOT in the top 3 by pass involvement: 111 / 256 (43.4%)


In [6]:
# Q5: how often is the targeted player also #1 by betweenness centrality (within their team-match)?
bt_rank = player_network_baseline.copy()
bt_rank['_bt_rank'] = bt_rank.groupby(['match_id', 'team'])['betweenness_centrality'].rank(
    ascending=False, method='min'
)
merged = team_match_robustness.merge(
    bt_rank[['match_id', 'team', 'player', '_bt_rank']],
    left_on=['match_id', 'team', 'targeted_player'],
    right_on=['match_id', 'team', 'player'],
    how='left',
)
top1_by_betweenness = (merged['_bt_rank'] == 1).sum()
print(f"Q5. Targeted player is also #1 by betweenness centrality: "
      f"{top1_by_betweenness} / {n} ({top1_by_betweenness/n:.1%})")
print()
print("=> Structural vulnerability is frequently NOT the player with the ball the most,")
print("   and frequently not even the highest-betweenness player by the standard metric.")

Q5. Targeted player is also #1 by betweenness centrality: 58 / 256 (22.7%)

=> Structural vulnerability is frequently NOT the player with the ball the most,
   and frequently not even the highest-betweenness player by the standard metric.


### Absolute / standardized concentration measures

`top1_top2_gap` and `targeted_standardized_gap` = (targeted − eligible mean) / eligible std
are the measures to use going forward — not `targeted_to_random_ratio`, which is unstable
whenever the eligible-pool mean is near zero (105 / 256 team-matches here).

In [7]:
summary_cols = ['targeted_efficiency_damage', 'eligible_efficiency_damage_mean',
                'eligible_efficiency_damage_std', 'targeted_excess_efficiency_damage',
                'targeted_standardized_gap', 'top1_top2_gap']
team_match_robustness[summary_cols].describe().round(4)

,targeted_efficiency_damage,eligible_efficiency_damage_mean,eligible_efficiency_damage_std,targeted_excess_efficiency_damage,targeted_standardized_gap,top1_top2_gap
count,256.0000,256.0000,256.0000,256.0000,256.0000,256.0000
mean,0.0390,0.0032,0.0188,0.0358,1.7792,0.0166
std,0.0285,0.0048,0.0055,0.0245,0.5882,0.0266
min,0.0152,-0.0000,0.0080,0.0125,0.9755,0.0000
25%,0.0236,0.0000,0.0154,0.0229,1.4004,0.0000
50%,0.0277,0.0002,0.0174,0.0265,1.5757,0.0073
75%,0.0357,0.0043,0.0195,0.0331,1.9846,0.0139
max,0.1282,0.0258,0.0397,0.1153,3.4555,0.1074


In [8]:
gap_median = team_match_robustness['top1_top2_gap'].median()
gap_p75 = team_match_robustness['top1_top2_gap'].quantile(0.75)
z_median = team_match_robustness['targeted_standardized_gap'].median()
z_p75 = team_match_robustness['targeted_standardized_gap'].quantile(0.75)

print(f"Median top1-top2 gap: {gap_median:.4f}  |  75th percentile: {gap_p75:.4f}")
print(f"Median standardized gap (z-score): {z_median:.2f}  |  75th percentile: {z_p75:.2f}")
print()
print("Most team-matches show a fairly flat top of the distribution (small median gap),")
print("but the 75th-percentile and max show real cases of one player standing out sharply —")
print("consistent with concentrated vulnerability being the exception, not the rule.")

Median top1-top2 gap: 0.0073  |  75th percentile: 0.0139
Median standardized gap (z-score): 1.58  |  75th percentile: 1.98

Most team-matches show a fairly flat top of the distribution (small median gap),
but the 75th-percentile and max show real cases of one player standing out sharply —
consistent with concentrated vulnerability being the exception, not the rule.


## 4. Illustrative examples (not case studies, not causal claims)

### A. 5 team-matches with the largest standardized gap (most reliable "stands out" measure)

In [9]:
example_cols = [
    'team', 'year', 'targeted_player', 'targeted_efficiency_damage',
    'eligible_efficiency_damage_mean', 'targeted_standardized_gap', 'top1_top2_gap',
    'targeted_pass_involvement_rank',
]

team_match_robustness.sort_values('targeted_standardized_gap', ascending=False)[example_cols].head(5)

,team,year,targeted_player,targeted_efficiency_damage,eligible_efficiency_damage_mean,targeted_standardized_gap,top1_top2_gap,targeted_pass_involvement_rank
166,Iran,2022,Mehdi Taremi,0.116975,0.008712,3.455474,0.089317,2
153,France,2022,Aurélien Djani Tchouaméni,0.121199,0.010031,3.385509,0.100626,1
133,Switzerland,2022,Granit Xhaka,0.124434,0.010019,3.334877,0.101810,1
225,United States,2022,Tyler Adams,0.111111,0.008754,3.331775,0.086580,1
241,Netherlands,2022,Andries Noppert,0.087199,0.008782,3.311981,0.068646,5


### B. 5 team-matches with the largest top1-top2 gap

In [10]:
team_match_robustness.sort_values('top1_top2_gap', ascending=False)[example_cols].head(5)

,team,year,targeted_player,targeted_efficiency_damage,eligible_efficiency_damage_mean,targeted_standardized_gap,top1_top2_gap,targeted_pass_involvement_rank
121,France,2018,N'Golo Kanté,0.128217,0.013617,2.884147,0.107438,2
133,Switzerland,2022,Granit Xhaka,0.124434,0.010019,3.334877,0.101810,1
153,France,2022,Aurélien Djani Tchouaméni,0.121199,0.010031,3.385509,0.100626,1
112,France,2018,Paul Pogba,0.126932,0.011632,3.165475,0.100442,2
246,Morocco,2022,Yassine Bounou,0.115034,0.011594,3.215438,0.098330,5


## Next step

Part 3 output is cached to `data/processed/team_match_robustness.csv`. Part 4 (two-player
combination search) will use the same eligibility pool and `efficiency_damage` metric to
test whether joint removal reveals interaction effects beyond the individually most
damaging players.